### https://docs.ray.io/en/latest/cluster/vms/user-guides/community/spark.html


Config:
- 2 Workers: 110 GB Memory, 16 Cores
- 1 Driver: 220 GB Memory, 32 Cores
Runtime:
- 16.4 LTS ML
Type:
- Standard_NC16as_T4_v3 [T4]


Notes: 

- We recommend setting the argument num_cpus_worker_node to the number of CPU cores per Apache Spark worker node. Similarly, setting num_gpus_worker_node to the number of GPUs per Apache Spark worker node is optimal. With this configuration, each Apache Spark worker node launches one Ray worker node that will fully utilize the resources of each Apache Spark worker node.
- Set the environment variable RAY_memory_monitor_refresh_ms to 0 within the Databricks cluster configuration when starting your Apache Spark cluster.


- In each spark worker node, we recommend making the sum of 'spark_executor_memory + num_Ray_worker_nodes_per_spark_worker * (memory_worker_node + object_store_memory_worker_node)' to be less than 'spark_worker_physical_memory * 0.8', otherwise it might lead to spark worker physical memory exhaustion and Ray task OOM errors.

In [0]:
# You configured 'spark.task.resource.gpu.amount' to 1.0,we recommend setting this value to 0 so that Spark jobs do not reserve GPU resources, preventing Ray-on-Spark workloads from having the maximum number of GPUs available. 

spark.conf.set("spark.task.resource.gpu.amount", "0")

In [0]:
import yaml

config_path = "../local_config.yaml"
with open(config_path, 'r') as config_file:
    config = yaml.safe_load(config_file)

In [0]:
import os
os.environ['HF_DATASETS_CACHE'] = config.get("cifar_cache")

In [0]:
%sh nvidia-smi

In [0]:
from utils import hf_dataset_utilities as hf_util

cifar_dataset = hf_util.hfds_download_volume(
    hf_cache = os.environ['HF_DATASETS_CACHE'],
    dataset_path= 'uoft-cs/cifar10',
    trust_remote_code = True, 
    disable_progress = False, 
    )
CIFARDataset = hf_util.create_torch_image_dataset(
    image_key="img",
    label_key="label"
    )

ds_transforms = hf_util.default_image_transforms(
    image_size = 32, 
    normalize_transform=True, 
    convert_rgb=True
    )

train_dataset = CIFARDataset(cifar_dataset['train'], transform=ds_transforms)
test_dataset = CIFARDataset(cifar_dataset['test'], transform=ds_transforms)

In [0]:
import ray

total_cores = int(spark.sparkContext.defaultParallelism)
num_workers = int(ray.util.spark.MAX_NUM_WORKER_NODES)
total_gpus = int(spark.sparkContext.getConf().get("spark.driver.resource.gpu.amount"))

print(f"Total cores: {total_cores}")
print(f"Total GPUs: {total_gpus}")
print(f"Total workers: {num_workers}")

In [0]:
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

setup_ray_cluster(
  # Set min and max to disable autoscaling
  min_worker_nodes=2, 
  max_worker_nodes=2,
  num_cpus_per_node=total_cores,
  num_gpus_per_node=total_gpus,
  num_cpus_head_node=total_cores,
  num_gpus_head_node=total_gpus,
  collect_log_to_path="/dbfs/tmp/ray_collected_logs"
)

# Pass any custom Ray configuration with ray.init
ray.init(ignore_reinit_error=True)

In [0]:
import os
import tempfile

import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor, Normalize, Compose

import ray.train.torch

def train_func():
    # Model, Loss, Optimizer
    model = resnet18(num_classes=10)
    # [1] Prepare model.
    model = ray.train.torch.prepare_model(model)
    # model.to("cuda")  # This is done by `prepare_model`
    criterion = CrossEntropyLoss()

    learning_rate = 1e-5
    betas = (0.9, 0.999)
    epsilon = 1e-08
    optimizer = Adam(model.parameters(), lr=learning_rate, betas=betas, eps=epsilon, weight_decay=0)

    # Use dataset shards passed through datasets parameter
    # train_dataset = ray.train.get_dataset_shard("train")
    # train_loader = train_dataset_shard.iter_torch_batches(
    #     batch_size=256, dtypes=torch.float
    # )

    # [2] Prepare dataloader.
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    train_loader = ray.train.torch.prepare_data_loader(train_loader)

    # Training
    for epoch in range(10):
        if ray.train.get_context().get_world_size() > 1:
            train_loader.sampler.set_epoch(epoch)

        for images, labels in train_loader:
            # This is done by `prepare_data_loader`!
            # images, labels = images.to("cuda"), labels.to("cuda")
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # [3] Report metrics and checkpoint.
        metrics = {"loss": loss.item(), "epoch": epoch}
        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
             # Save model state_dict based on whether it's wrapped in DataParallel or not
            if isinstance(model, torch.nn.DataParallel) or isinstance(model, torch.nn.parallel.DistributedDataParallel):
                torch.save(
                    model.module.state_dict(),
                    os.path.join(temp_checkpoint_dir, "model.pt")
                )
            else:
                torch.save(
                    model.state_dict(),
                    os.path.join(temp_checkpoint_dir, "model.pt")
                )

            ray.train.report(
                metrics,
                checkpoint=ray.train.Checkpoint.from_directory(temp_checkpoint_dir),
            )
        if ray.train.get_context().get_world_rank() == 0:
            print(metrics)

In [0]:

from ray.train import RunConfig

#  [4] Configure scaling and resource requirements.
# Use GPU to allow cuda 
scaling_config = ray.train.ScalingConfig(num_workers=1, use_gpu=True)

# Local path (/some/local/path/unique_run_name)
run_config = RunConfig(storage_path="/dbfs/tmp/ray_ws_logs", name="local")

ray_train_ds = ray.data.from_torch(train_dataset)
ray_test_ds = ray.data.from_torch(test_dataset)

datasets = {
    "train": train_dataset,
    "test": test_dataset
}

# [5] Launch distributed training job.
trainer = ray.train.torch.TorchTrainer(
    train_func,
    scaling_config=scaling_config,
    # [5a] If running in a multi-node cluster, this is where you
    # should configure the run's persistent storage that is accessible
    # across all worker nodes.
    run_config=run_config,
    # datasets=datasets,
)

result = trainer.fit()

In [0]:
display(result.metrics)     # The metrics reported during training.
display(result.checkpoint)  # The latest checkpoint reported during training.
display(result.path)        # The path where logs are stored.
display(result.error)       # The exception that was raised, if training failed.

In [0]:
# [6] Load the trained model.
with result.checkpoint.as_directory() as checkpoint_dir:
    model_state_dict = torch.load(os.path.join(checkpoint_dir, "model.pt"))
    model = resnet18(num_classes=10)
    model.load_state_dict(model_state_dict)

In [0]:
ray.util.spark.shutdown_ray_cluster()